# Xenium Run 1 – 01 Quality control and section assignment

**Input:** Xenium Ranger re-segmentation output of slide 0028164 (`test_segmentation4_0/outs`, Xenium Ranger `resegment`, multimodal cell segmentation).
**Output:** `{DATA_DIR}/20251009_Run1_reSeg_QC.h5ad` – QC-filtered cells, log-normalised expression (raw counts in `layers["counts"]`), section (`timepoint`) and a first major cell-type annotation (`compartment_annotation`).

Steps
1. Load the re-segmented Xenium bundle
2. Cell-level QC (transcripts, detected genes, cell area) and gene filter
3. Normalisation, PCA, UMAP, Leiden (for overview only)
4. Assignment of every cell to one of the 10 lymph-node sections on the slide (`timepoint`)
5. First major cell-type annotation by marker-gene module scores (highest score wins)

Next notebook: `Xenium_Run1_02_annotation.ipynb`

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import seaborn as sns
import matplotlib.pyplot as plt
from spatialdata_io import xenium

In [ ]:
# ------------------------------------------------------------
# Paths relative to this folder (adapt BASE_DIR if your data live elsewhere)
# ------------------------------------------------------------
BASE_DIR = ".."
RAW_DIR  = f"{BASE_DIR}/01_Raw"
DATA_DIR = f"{BASE_DIR}/02_ProcessedData"

XENIUM_PATH = f"{RAW_DIR}/test_segmentation4_0/outs/"                 # re-segmented Xenium bundle
QC_OUT      = f"{DATA_DIR}/20251009_Run1_reSeg_QC.h5ad"  # output of this notebook

## 2. Load data

In [ ]:
sdata = xenium(XENIUM_PATH)
adata = sdata.tables["table"]
adata

## 3. Cell-level QC

QC metrics and fraction of negative-control probes / codewords.

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

cprobes = (
    adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
cwords = (
    adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
print(f"Negative DNA probe count % : {cprobes}")
print(f"Negative decoding count % : {cwords}")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata.obs["total_counts"],
    kde=False,
    ax=axs[0],
)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1],
)


axs[2].set_title("Area of segmented cells")
sns.histplot(
    adata.obs["cell_area"],
    kde=False,
    ax=axs[2],
)

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3],
)

### Filter cells and genes

* cell area 20–80 µm² (removes debris / partial segmentation and merged cells)
* ≥ 80 transcripts and ≥ 10 detected genes (429-gene panel, small resting lymphocytes)
* ≤ 99.5th percentile of transcripts and detected genes
* genes detected in ≥ 100 cells

In [ ]:
# Plot histogram again to confirm distribution
sns.histplot(adata.obs["cell_area"], bins=500)
plt.axvline(20, color="r", ls="--", label="min_area")
plt.axvline(80, color="r", ls="--", label="max_area")
plt.legend()
plt.title("Cell area (µm²)")
plt.show()

min_area = 20    # debris / partial segmentation
max_area = 80   # merged or too large

# keep previous transcript-based filters but slightly lower total_counts threshold
min_counts = 80    # Xenium 429-gene panel → few transcripts per small lymphocyte
max_counts = np.quantile(adata.obs["total_counts"], 0.995)
min_genes  = 10    # allow small resting lymphocytes
max_genes  = np.quantile(adata.obs["n_genes_by_counts"], 0.995)

qc_mask = (
    (adata.obs["total_counts"] >= min_counts)
    & (adata.obs["total_counts"] <= max_counts)
    & (adata.obs["n_genes_by_counts"] >= min_genes)
    & (adata.obs["n_genes_by_counts"] <= max_genes)
    & (adata.obs["cell_area"] >= min_area)
    & (adata.obs["cell_area"] <= max_area)
)
adata_qc = adata[qc_mask].copy()
sc.pp.filter_genes(adata_qc, min_cells=100)
print(f"Cells retained: {qc_mask.sum()} / {adata.n_obs}")
adata_qc

In [ ]:
adata = adata_qc.copy()
del adata_qc

## 4. Normalisation and low-dimensional embedding

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata)

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "total_counts",
        "n_genes_by_counts",
        "leiden",
    ],
    wspace=0.4,
)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color=[
        "leiden",
    ],
    wspace=0.4,
)

## 5. Assign cells to lymph-node sections

10 lymph-node sections were placed on slide 0028164. Each cell is assigned to its section by its
x/y position (`timepoint` = `<section>:Day<d>_<mouse ID>`); cells outside all sections are `Unknown`.
Colours and sample order used throughout.

In [ ]:
ln_major_colors = {
    "B_cells": "#4A90E2",        # soft blue
    "T_cells": "#50E3C2",        # gentle green
    "Tg_T_cells": "#D0021B",     # magenta / highlight
    "Macrophages": "#F5A623",    # warm gold
    "Stromal_cells": "#9B9B9B",  # neutral gray-violet
    "Lymphatic_EC": "#2E8B57",    # teal-cyan,
    "Tregs": "#7B6FD0",          # muted violet
    "cDC1": "#084594",           # deep navy blue (clearly blue)
    "cDC2": "#6B8E23",           # olive / yellow-green (clearly green)
    "GC_B_cells": "#6A3D9A"      # deep purple
}
ordered_timepoints = [
    "10:Day0_0382",
    "2:Day2_0386", "8:Day2_0385", "9:Day2_0385",
    "4:Day3_0388", "6:Day3_0388", "7:Day3_0388",
    "3:Day4_0391", "5:Day4_0391",
    "1:Day6_0394"
]

In [ ]:
adata.obs["spatial1"] = adata.obsm["spatial"][:, 0]
adata.obs["spatial2"] = adata.obsm["spatial"][:, 1]


def assign_ln(row):
    #1
    if row["spatial1"] > 4000 and row["spatial2"] > 17500:
        return "1:Day6_0394"
    #2
    elif row["spatial1"] < 4000 and row["spatial2"] > 15000:
        return "2:Day2_0386"
    #3
    elif row["spatial1"] > 4000 and row["spatial2"] > 13500 and row["spatial2"] < 17000:
        return "3:Day4_0391"
    #4
    elif row["spatial1"] < 4000 and row["spatial2"] > 10000 and row["spatial2"] < 12500:
        return "4:Day3_0388"
    #5
    elif row["spatial1"] > 6000 and row["spatial2"] > 10000 and row["spatial2"] < 13500:
        return "5:Day4_0391"
    #6
    elif row["spatial1"] < 4500 and row["spatial2"] > 5000 and row["spatial2"] < 8000:
        return "6:Day3_0388"
    #7
    elif row["spatial1"] > 6000 and row["spatial2"] > 6000 and row["spatial2"] < 9500:
        return "7:Day3_0388"
    #8
    elif row["spatial1"] < 5500 and row["spatial2"] < 2500:
        return "8:Day2_0385"
    #9
    elif row["spatial1"] > 4500 and row["spatial1"] < 10000 and row["spatial2"] > 2500 and row["spatial2"] < 8000:
        return "9:Day2_0385"
    #10
    elif row["spatial1"] > 5000 and row["spatial2"] < 2500:
        return "10:Day0_0382"
    else:
        return "Unknown"

adata.obs["timepoint"] = adata.obs.apply(assign_ln, axis=1)

adata.uns.pop("timepoint_colors", None)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color="timepoint",
    wspace=0.4,
)

## 6. First major cell-type annotation

Module score per major cell type (`sc.tl.score_genes`); every cell gets the type with the highest score
(`compartment_annotation`). This annotation is refined in notebook 02.

In [ ]:
ln_major_markers = {
    "B_cells": ["Cd79a","Cr2", "Cd38", "Cxcr5"],
    "T_cells": ["Cd3d", "Ccr7", "Cd4", "Cd8a"],
    "Tg_T_cells": ["ENA_Y11102_Y11102.1", "ENA_Y11103_Y11103.1", "CD45.1", "Cd3d", "Cd4"],
    "Macrophages": ["Cd68", "Sirpa"],
    "Stromal_cells": ["Timp3", "Cfh", "Cd21", "Cxcl13", "Ccl21"],
    "Lymphatic_EC": ["Lyve1", "Prox1", "Plvap"]
}

In [ ]:
# 1. Compute gene set scores
for comp, genes in ln_major_markers.items():
    sc.tl.score_genes(adata, gene_list=genes, score_name=f"{comp}_score")

# 2. Get compartment list
compartments = list(ln_major_markers.keys())

# 3. Annotate each cell by its highest score
adata.obs["compartment_annotation"] = adata.obs[
    [f"{c}_score" for c in compartments]
].idxmax(axis=1).str.replace("_score", "")

In [ ]:
adata.obs["compartment_annotation"] = adata.obs["compartment_annotation"].astype("category")
adata.uns["compartment_annotation_colors"] = [
    ln_major_colors[c] for c in adata.obs["compartment_annotation"].cat.categories
]
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color=[
        "compartment_annotation",
    ],
    wspace=0.4,
)

### Composition per section

In [ ]:
# Remove entries with "Unknown" timepoint
adata_filtered = adata[adata.obs["timepoint"] != "Unknown"].copy()

# Compute counts per timepoint and cell type
cell_fractions = (
    adata_filtered.obs.groupby(["timepoint", "compartment_annotation"])
    .size()
    .reset_index(name="count")
)

# Normalize per timepoint → percentage
cell_fractions["fraction"] = (
    cell_fractions.groupby("timepoint")["count"].transform(lambda x: x / x.sum() * 100)
)

# Ensure consistent ordering
cell_fractions["timepoint"] = pd.Categorical(
    cell_fractions["timepoint"],
    categories=ordered_timepoints,
    ordered=True
)

In [ ]:
pivot_df = cell_fractions.pivot_table(index='timepoint', columns='compartment_annotation', values='fraction', fill_value=0)

# Plot stacked barplot
pivot_df.plot(kind='bar', stacked=True, color=[ln_major_colors.get(c, "#7f7f7f") for c in pivot_df.columns], figsize=(8,3))
#pivot_df.plot(kind='bar', stacked=True,  figsize=(8,3))
plt.ylabel("Count")
plt.xlabel("Timepoint")
plt.title("Stacked Barplot of Cell Fractions per Superclass")
plt.xticks(rotation=45)
plt.legend(title="Superclass", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 7. Save

In [ ]:
adata.write(QC_OUT)
print(adata.shape, '->', QC_OUT)